# 1. Data Ingestion and Connectors

In a real Security Operations Center (SOC), step one is **getting data into the SIEM**. In Microsoft Sentinel this is done with *data connectors*; in our mini-SIEM it is one REST endpoint (`POST /ingest`) that writes to named tables.

### What you'll learn
- How log data lands in a SIEM (the "data connector" idea)
- Why raw log files are **not** enough for security investigations — the bad → best progression
- How to query logs by table, filter, time window, and aggregation (KQL-like)
- How to build your own custom connector

## 0. Setup — pick the lab kernel

This lab has its own `uv`-managed virtual environment. Before running any code cell:

1. From `security-certs/sc-200/01-build-a-siem/` run once in a terminal:
   ```bash
   uv sync
   docker compose up -d
   ```
2. In VS Code, click the kernel picker (top-right of this notebook) and choose **`.venv (Python 3.xx)`** from this folder.
3. If the kernel does not appear, reload the window: `Cmd+Shift+P` → `Reload Window`.

The log-generator container has already seeded the SIEM with normal traffic **and** four attack patterns (brute force, lateral movement, exfiltration, phishing). Every cell below talks to `http://localhost:8000`.

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

def pp(r):
    print(json.dumps(r.json(), indent=2))

# Sanity-check: the SIEM is reachable and already has data
print('=== SIEM Dashboard ===')
dash = httpx.get(f'{SIEM}/dashboard').json()
print(json.dumps(dash, indent=2))

# Every cell below reads the seeded dataset. If the log generator did not finish,
# the notebook would otherwise print empty lists and look like it "worked".
EXPECTED_TABLES = {'SigninLogs', 'AzureFirewall', 'DeviceEvents', 'EmailEvents'}
assert dash['total_logs'] > 0, 'SIEM has no logs - run `docker compose up -d` and wait ~10s for sc200-log-generator'
missing = EXPECTED_TABLES - set(dash['tables'])
assert not missing, f'seed data incomplete, missing tables: {sorted(missing)}'
assert dash['active_rules'] >= 6, f"expected the 6 seeded analytics rules, found {dash['active_rules']}"


## 1.1 The "bad" way — reading raw log files

Before SIEMs existed, investigators would SSH into a server and `grep` log files. It *sort of* works for one box, but it breaks down fast:

- 🔴 **One host at a time** — can't see attacks that span firewall + endpoint + email
- 🔴 **No structure** — every product logs differently (CSV, JSON, free text)
- 🔴 **No history** — log rotation deletes yesterday's evidence
- 🔴 **Easy to miss** — humans can't eyeball 10 million lines an hour

Let's *simulate* the bad way first so the SIEM benefit is obvious.

In [ ]:
# The bad way: grep a plain text log file on one box.
# We write a small fake syslog locally so you can feel the pain.
from pathlib import Path
import re

log_path = Path('/tmp/fake_sshd.log')
log_path.write_text('\n'.join([
    'Apr 20 10:01:02 vm-web-01 sshd[1]: Accepted password for bob from 10.0.1.11',
    'Apr 20 10:05:17 vm-web-01 sshd[2]: Failed password for alice from 185.220.101.42',
    'Apr 20 10:05:18 vm-web-01 sshd[3]: Failed password for alice from 185.220.101.42',
    'Apr 20 10:05:19 vm-web-01 sshd[4]: Failed password for alice from 185.220.101.42',
    'Apr 20 10:06:02 vm-web-01 sshd[5]: Accepted password for alice from 185.220.101.42',
]))

# Count failed logins per user with grep-style code:
failures = {}
for line in log_path.read_text().splitlines():
    m = re.search(r'Failed password for (\S+) from (\S+)', line)
    if m:
        failures[m.group(1)] = failures.get(m.group(1), 0) + 1
print('grep-style result:', failures)
print('\nProblems with this approach:')
print('  - only sees THIS file on THIS box')
print('  - parses one log format; another product = rewrite the regex')
print('  - no time windows, no correlation, no alerting')

## 1.2 The "best" way — a SIEM with typed tables

A SIEM normalises every data source into a **table** with known columns. Ours has four, matching common Sentinel tables:

| Our table | Real Sentinel table | What it contains |
|-----------|--------------------|-----------------|
| `SigninLogs` | `SigninLogs` | Entra ID sign-in events |
| `AzureFirewall` | `AzureDiagnostics` (filtered) | Firewall allow/deny decisions |
| `DeviceEvents` | `DeviceEvents` | Endpoint process executions, file operations |
| `EmailEvents` | `EmailEvents` | Email delivery, phishing detection |

One query API works across **all** tables and **all** hosts — that's the SIEM superpower.

In [ ]:
# Query sign-in logs — structured, not free-text
print('=== Recent Sign-in Logs (last 5) ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'limit': 5,
})
for log in r.json()['results']:
    status = '✅' if log['ResultType'] == 'Success' else '❌'
    print(f'  {status} {log["UserPrincipalName"]:<25} {log["IPAddress"]:<16} {log["Location"]:<18} {log["AppDisplayName"]}')

In [ ]:
# Firewall: filter by column, no regex needed
print('=== Firewall: traffic to a known-bad IP ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'AzureFirewall',
    'filter': {'DestinationIP': '185.220.101.42'},
    'limit': 20,
})
rows = r.json()['results']
for log in rows[:5]:
    mb = log.get('BytesSent', 0) / 1e6
    print(f'  ⚠️  {log["SourceIP"]} → {log["DestinationIP"]}:{log["DestinationPort"]} [{log["Action"]}]  {mb:,.0f} MB')

# A filter that silently returns nothing looks identical to a filter that found
# nothing suspicious. Assert, so a broken seed fails here instead of three cells later.
assert len(rows) >= 3, f'expected the seeded exfiltration flows to this IP, got {len(rows)}'
print(f'\n{len(rows)} flows, {sum(l.get("BytesSent", 0) for l in rows)/1e6:,.0f} MB total, all to one external address.')


## 1.3 Aggregation — count failures per user (KQL-style)

In real Sentinel KQL this is:

```kql
SigninLogs
| where TimeGenerated > ago(1h)
| where ResultType != 0          // 0 == success; every other code is a failure reason
| summarize FailCount = count() by UserPrincipalName
| order by FailCount desc
```

> ⚠️ **Do not write `where ResultType == "Failure"`.** The real `SigninLogs.ResultType`
> is the Entra **sign-in error code as a string** — `"0"` for success, `"50126"` for bad
> credentials, `"50053"` for smart lockout, `"53003"` for a Conditional Access block.
> There is no `"Failure"` value in the schema, so that rule matches nothing and never
> fires. The human-readable text lives in `ResultDescription`.
>
> Our mini-SIEM stores the *normalised* values `Success` / `Failure` — that normalisation
> step is exactly what a real data connector does for you, and it is why the queries
> below read `ResultType: Failure` while the KQL above reads `ResultType != 0`.

Our mini-SIEM does the same aggregation with `aggregate_by`:

In [ ]:
# The seeded "Brute force sign-in" rule fires at >= 5 failures per user in 1 hour,
# so flag at exactly that threshold - a chart that disagrees with the rule teaches
# the wrong number.
BRUTE_FORCE_THRESHOLD = 5

print('=== Failed sign-ins by user ===')
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'filter': {'ResultType': 'Failure'},
    'aggregate_by': 'UserPrincipalName',
})
by_user = {row['group_key']: row['count'] for row in r.json()['results']}
for user, count in sorted(by_user.items(), key=lambda kv: -kv[1]):
    bar = '█' * min(count, 30)
    alert = f' ⚠️ >= {BRUTE_FORCE_THRESHOLD}, would alert' if count >= BRUTE_FORCE_THRESHOLD else ''
    print(f'  {user:<30} {count:>3} {bar}{alert}')

# The point of the chart is the CONTRAST: most users have a couple of typos, one has
# a brute force. If everyone looked the same there would be nothing to detect.
over = [u for u, c in by_user.items() if c >= BRUTE_FORCE_THRESHOLD]
assert len(by_user) >= 3, f'need a benign baseline to compare against, only saw {len(by_user)} user(s)'
assert over == ['alice@contoso.com'], f'expected only alice over the threshold, got {sorted(over)}'
print(f'\n{len(by_user)} users had failures; {len(over)} crossed the alerting threshold.')
print('Everyone else is normal password-typo noise - and that noise is what makes')
print('the threshold a real decision instead of a number nobody can be wrong about.')


## 1.4 Custom data connector

Sentinel ships ~100 built-in connectors, but almost every org has a **custom app** that logs to its own format. The answer is: **push JSON** into a custom table. In real Sentinel this is the Logs Ingestion API + a Data Collection Rule (DCR). In our mini-SIEM it is `POST /ingest/batch`.

In [ ]:
custom_logs = [
    {'table_name': 'CustomAppLogs', 'data': {'action': 'login', 'user': 'admin', 'ip': '10.0.1.50', 'status': 'success'}},
    {'table_name': 'CustomAppLogs', 'data': {'action': 'login', 'user': 'admin', 'ip': '185.220.101.42', 'status': 'failure'}},
    {'table_name': 'CustomAppLogs', 'data': {'action': 'export_data', 'user': 'admin', 'records': 50000, 'destination': 'external'}},
]

r = httpx.post(f'{SIEM}/ingest/batch', json={'entries': custom_logs})
print(f'Ingested {r.json()["count"]} entries into CustomAppLogs')

r = httpx.post(f'{SIEM}/query', json={'table_name': 'CustomAppLogs', 'limit': 10})
for log in r.json()['results']:
    flag = '🚩' if log.get('ip') == '185.220.101.42' or log.get('records', 0) > 10000 else '  '
    print(f'  {flag} {log}')

## Real Sentinel data connectors (exam reference)

In the SC-200 exam you need to know **which connector to use**:

| Data source | Connector type | Collection method |
|------------|----------------|-------------------|
| Entra ID sign-in logs | Built-in (1-click) | Direct API |
| Microsoft 365 | Built-in | Direct API |
| Azure Activity | Diagnostic settings | Azure Monitor pipeline |
| Windows Security Events | AMA (Azure Monitor Agent) | Data Collection Rule (DCR) |
| Linux Syslog | AMA + Syslog | Data Collection Rule |
| CEF (third-party firewalls) | AMA + CEF | Log forwarder → DCR |
| Custom application logs | Logs Ingestion API | REST API + DCR |

### Exam tip: AMA vs legacy agents

- **AMA (Azure Monitor Agent)** — current standard. Uses DCRs. Supports multi-homing.
- **Log Analytics agent (MMA)** — deprecated. Still tested for migration scenarios.
- **CEF/Syslog via AMA** — requires a Linux forwarder VM with the AMA extension.
- **DCR transformations** — KQL that runs at *ingest* time (drop noisy fields, lower cost).

### Cost reality check

Sentinel charges per GB ingested. Two levers matter:
1. **Filter at source** (e.g., don't send Verbose sysmon).
2. **Use Basic/Auxiliary log tiers** for high-volume, low-signal logs (e.g., firewall allow-all).

**Next**: [Notebook 2 — Analytics Rules and Detection](02_analytics_rules.ipynb)